# 🏋️ Fun with Text and Image Embeddings 🍎

Welcome to our **Health & Fitness** embeddings notebook! In this tutorial, we'll show you how to:

1. **Initialize** an `AIProjectClient` to access your Azure AI Foundry project.
2. **Embed text** using `azure-ai-openai` with our fun health-themed phrases.
3. **Use a prompt template** for extra context.

Let's get started and have some fun with our healthy ideas! 🍏

> **Disclaimer**: This notebook is for educational purposes only. Always consult a professional for medical advice.

<img src="./seq-diagrams/2-embeddings.png" width="30%"/>

## 1. Setup & Environment

#### Prerequisites:
- Deploy a text embeddings model (**text-embedding-3-large**) in Azure AI Foundry, already performed in Exercise 1, Task 1.

#
We'll import our libraries and load the environment variables for:
- `PROJECT_ENDPOINT`: Your Azure AI Foundry project endpoint.
- `EMBEDDING_MODEL_DEPLOYMENT_NAME`: The text embeddings model deployment name.

We'll import libraries, load environment variables, and create an `AIProjectClient`.

> #### Complete [1-basic-chat-completion.ipynb](./1-basic-chat-completion.ipynb) notebook before starting this one
Let's begin! 🚀

In [ ]:
import os
from pathlib import Path
from urllib.parse import urlparse
from dotenv import load_dotenv
from openai import AzureOpenAI
from azure.identity import AzureCliCredential
from azure.ai.projects import AIProjectClient
import requests

# Load environment variables from workspace root .env
notebook_path = Path().absolute()
load_dotenv(notebook_path.parent.parent / '.env')

# Initialize credentials using AzureCliCredential
credential = AzureCliCredential()

# Parse PROJECT_ENDPOINT into required AIProjectClient constructor components
_url            = os.getenv("PROJECT_ENDPOINT")
_parsed         = urlparse(_url)
base_endpoint   = f"{_parsed.scheme}://{_parsed.netloc}"
path_parts      = [p for p in _parsed.path.split("/") if p]
project_name    = path_parts[-1] if path_parts else ""
hub_name        = _parsed.netloc.split(".")[0]
EMBEDDING_MODEL_DEPLOYMENT_NAME = os.getenv("EMBEDDING_MODEL_DEPLOYMENT_NAME", "text-embedding-3-large")

# Auto-detect subscription_id & resource_group from Foundry hub
print("Auto-detecting subscription ID and resource group...")
try:
    mgmt_token = credential.get_token("https://management.azure.com/.default").token
    headers = {"Authorization": f"Bearer {mgmt_token}"}

    # List all accessible subscriptions
    subs = requests.get(
        "https://management.azure.com/subscriptions?api-version=2020-01-01",
        headers=headers, timeout=15
    ).json().get("value", [])

    subscription_id = None
    resource_group = None
    
    for sub in subs:
        sub_id = sub["subscriptionId"]
        # Search for the Foundry hub: type=Microsoft.CognitiveServices/accounts, name=hub_name
        resources = requests.get(
            f"https://management.azure.com/subscriptions/{sub_id}/resources"
            f"?$filter=name eq '{hub_name}' and "
            f"resourceType eq 'Microsoft.CognitiveServices/accounts'"
            f"&api-version=2021-04-01",
            headers=headers, timeout=15
        ).json().get("value", [])

        if resources:
            # ARM resource ID: /subscriptions/<sub>/resourceGroups/<rg>/providers/...
            rg_from_id = resources[0]["id"].split("/")[4]
            subscription_id = sub_id
            resource_group = rg_from_id
            break

    if not (subscription_id and resource_group):
        raise RuntimeError(f"Hub '{hub_name}' not found in any accessible subscription.")

    print(f"✓ Subscription ID:  {subscription_id[:8]}...")
    print(f"✓ Resource group:   {resource_group}")

except Exception as e:
    raise EnvironmentError(
        f"Failed to auto-detect subscription and resource group: {e}"
    ) from e

try:
    project_client = AIProjectClient(
        endpoint=base_endpoint,
        subscription_id=subscription_id,
        resource_group_name=resource_group,
        project_name=project_name,
        credential=credential,
    )
    print("🎉 Successfully created AIProjectClient")
except Exception as e:
    print("❌ Error creating AIProjectClient:", e)

## 2. Text Embeddings

We'll call `client.embeddings.create()` from our `AzureOpenAI` to retrieve the embeddings client. Then we'll embed some fun health-themed phrases:

- "🍎 An apple a day keeps the doctor away"
- "🏋️ 15-minute HIIT workout routine"
- "🧘 Mindful breathing exercises"

The output will be numeric vectors representing each phrase in semantic space. Let’s see those embeddings!

In [ ]:
from openai import OpenAI

text_phrases = [
    "An apple a day keeps the doctor away 🍎",
    "Quick 15-minute HIIT workout routine 🏋️",
    "Mindful breathing exercises 🧘"
]

print(f"Using AZURE_OPENAI_ENDPOINT: {os.getenv('AZURE_OPENAI_ENDPOINT')}")
print(f"Embedding model: {EMBEDDING_MODEL_DEPLOYMENT_NAME}\n")

try:
    # Use OpenAI SDK with Azure OpenAI endpoint (where the embedding model is deployed)
    client = OpenAI(
        api_key=os.getenv("AZURE_OPENAI_KEY"),
        base_url=os.getenv("AZURE_OPENAI_ENDPOINT")
    )

    response = client.embeddings.create(
        model=EMBEDDING_MODEL_DEPLOYMENT_NAME,
        input=text_phrases
    )

    for item in response.data:
        vec = item.embedding
        sample_str = f"[{vec[0]:.4f}, {vec[1]:.4f}, ..., {vec[-2]:.4f}, {vec[-1]:.4f}]"
        print(
            f"Sentence {item.index}: '{text_phrases[item.index]}':\n"
            f"  Embedding length={len(vec)}\n"
            f"  Sample: {sample_str}\n"
        )

except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

## 3. Prompt Template Example 📝

Even though our focus is on embeddings, here's how you might prepend some context to a user message. Imagine you want to embed user text but first add a system prompt such as “You are HealthFitGPT, a fitness guidance model…” This little extra helps set the stage for more context-aware embeddings.


In [ ]:
# A basic prompt template (system-style) we'll prepend to user text.
TEMPLATE_SYSTEM = (
    "You are HealthFitGPT, a fitness guidance model.\n"
    "Please focus on healthy advice and disclaim you're not a doctor.\n\n"
    "User message:"  # We'll append the user message after this.
)

# Create OpenAI client using AZURE_OPENAI_ENDPOINT (where the embedding model is deployed)
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("AZURE_OPENAI_KEY"),
    base_url=os.getenv("AZURE_OPENAI_ENDPOINT")
)

def embed_with_template(user_text):
    content = TEMPLATE_SYSTEM + " " + user_text

    rsp = client.embeddings.create(
        model=EMBEDDING_MODEL_DEPLOYMENT_NAME,
        input=[content]
    )

    return rsp.data[0].embedding

sample_user_text = "Can you suggest a quick home workout for busy moms?"

try:
    embedding_result = embed_with_template(sample_user_text)
    print("Embedding length:", len(embedding_result))
    print("First few dims:", embedding_result[:8])
except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

## 4. Wrap-Up & Next Steps
🎉 We've shown how to:
- Set up the `AIProjectClient`.
- Get **text embeddings** using *text-embedding-3-large*.
- Use a **prompt template** to add system context to your embeddings.

**Where to go next?**
- Explore `azure-ai-evaluation` for evaluating your embeddings.
- Use `azure-core-tracing-opentelemetry` for end-to-end telemetry.
- Build out a retrieval pipeline to compare similarity of embeddings.

Have fun experimenting, and remember: when it comes to your health, always consult a professional!